# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# The .env file lives in this folder (project/starter) and contains:
# OPENAI_API_KEY="voc-**********"
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-********"
# See .env.example for the variable names. The .env file itself is git-ignored.
print("Using .env at:", os.path.abspath(".env"), "| exists:", os.path.exists(".env"))

Using .env at: /Users/evertongomes/project_udemy/project_AI_gaming_assistant/starter/.env | exists: True


In [4]:
from dotenv import find_dotenv

# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(usecwd=True), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
print("Environment variables loaded: OPENAI_API_KEY, OPENAI_BASE_URL, TAVILY_API_KEY")

Environment variables loaded: OPENAI_API_KEY, OPENAI_BASE_URL, TAVILY_API_KEY


### VectorDB Instance

In [5]:
# Persistent client: the database is stored in the "chromadb/" folder,
# which Part 2 loads with exactly the same path.
chroma_client = chromadb.PersistentClient(path="chromadb")
chroma_client

### Collection

In [6]:
# OpenAI embeddings, pointed at the key/base URL from .env.
# Part 2 rebuilds exactly the same embedding function when it loads the collection.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# Recreate the collection so the notebook can be re-run without duplicate IDs.
if "udaplay" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("udaplay")

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
collection

Collection(name=udaplay)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Index all the fields so semantic search can match on genre and publisher too
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']} "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}."
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print(f"Documents in collection '{collection.name}': {collection.count()}")

Documents in collection 'udaplay': 15


### Inspect the collection

In [9]:
# Peek at what was stored: id, indexed text and metadata
stored = collection.get(include=["documents", "metadatas"])
for doc_id, doc, meta in zip(stored["ids"], stored["documents"], stored["metadatas"]):
    print(f"{doc_id} | {meta['Name']} | {meta['Platform']} | {meta['YearOfRelease']} | {meta['Publisher']}")

001 | Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment
002 | Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | Rockstar Games
003 | Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment
004 | Marvel's Spider-Man | PlayStation 4 | 2018 | Sony Interactive Entertainment
005 | Marvel's Spider-Man 2 | PlayStation 5 | 2023 | Sony Interactive Entertainment
006 | Pokémon Gold and Silver | Game Boy Color | 1999 | Nintendo
007 | Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | Nintendo
008 | Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | Nintendo
009 | Super Mario 64 | Nintendo 64 | 1996 | Nintendo
010 | Super Smash Bros. Melee | GameCube | 2001 | Nintendo
011 | Wii Sports | Wii | 2006 | Nintendo
012 | Mario Kart 8 Deluxe | Nintendo Switch | 2017 | Nintendo
013 | Kinect Adventures! | Xbox 360 | 2010 | Microsoft Game Studios
014 | Minecraft | Xbox One | 2014 | Mojang Studios
015 | Halo Infinite | Xbox Series X|S | 2021 | Xbox Game 

### Semantic search

The questions below don't share exact keywords with the documents; the embeddings find the closest games by meaning.

In [10]:
def semantic_search(query: str, n_results: int = 3):
    """Query the vector DB and print the closest games with their distance (lower = more similar)."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["metadatas", "distances"],
    )
    print(f"\nQuery: {query}")
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        print(f"  - {meta['Name']} [{meta['Platform']}, {meta['YearOfRelease']}] "
              f"publisher={meta['Publisher']} (distance={dist:.3f})")
    return results


semantic_search("When was Pokémon Gold and Silver released?")
semantic_search("Which one was the first 3D platformer Mario game?")
semantic_search("Who published Grand Theft Auto: San Andreas?")
semantic_search("a realistic car racing simulator")
_ = semantic_search("Was Mortal Kombat X released for PlayStation 5?")


Query: When was Pokémon Gold and Silver released?
  - Pokémon Gold and Silver [Game Boy Color, 1999] publisher=Nintendo (distance=0.112)
  - Pokémon Ruby and Sapphire [Game Boy Advance, 2002] publisher=Nintendo (distance=0.147)
  - Super Mario 64 [Nintendo 64, 1996] publisher=Nintendo (distance=0.225)



Query: Which one was the first 3D platformer Mario game?
  - Super Mario 64 [Nintendo 64, 1996] publisher=Nintendo (distance=0.115)
  - Super Mario World [Super Nintendo Entertainment System (SNES), 1990] publisher=Nintendo (distance=0.140)
  - Mario Kart 8 Deluxe [Nintendo Switch, 2017] publisher=Nintendo (distance=0.205)



Query: Who published Grand Theft Auto: San Andreas?
  - Grand Theft Auto: San Andreas [PlayStation 2, 2004] publisher=Rockstar Games (distance=0.124)
  - Gran Turismo [PlayStation 1, 1997] publisher=Sony Computer Entertainment (distance=0.211)
  - Gran Turismo 5 [PlayStation 3, 2010] publisher=Sony Computer Entertainment (distance=0.216)



Query: a realistic car racing simulator
  - Gran Turismo [PlayStation 1, 1997] publisher=Sony Computer Entertainment (distance=0.119)
  - Gran Turismo 5 [PlayStation 3, 2010] publisher=Sony Computer Entertainment (distance=0.135)
  - Mario Kart 8 Deluxe [Nintendo Switch, 2017] publisher=Nintendo (distance=0.195)



Query: Was Mortal Kombat X released for PlayStation 5?
  - Marvel's Spider-Man 2 [PlayStation 5, 2023] publisher=Sony Interactive Entertainment (distance=0.210)
  - Marvel's Spider-Man [PlayStation 4, 2018] publisher=Sony Interactive Entertainment (distance=0.226)
  - Halo Infinite [Xbox Series X|S, 2021] publisher=Xbox Game Studios (distance=0.227)


### Reloading the persistent collection

This is how Part 2 loads the database: a new client on the same path and the same embedding function.

In [11]:
reloaded = chromadb.PersistentClient(path="chromadb").get_collection(
    "udaplay", embedding_function=embedding_fn
)
print("Reloaded collection:", reloaded.name, "| documents:", reloaded.count())

Reloaded collection: udaplay | documents: 15
